# 06 · Cross-Modal Sensor Consistency Check
Detects **silently-failing sensors** in production.

| Failure mode | Detector |
|---|---|
| Thermal camera miscalibration | Cross-modal cosine (thermal_emb vs tcn_emb) |
| Disconnected DXP probe | Flat-line variance detector |
| Noisy visual section | Embedding Mahalanobis anomaly score |

Unlike drift monitoring (which tracks distributions), consistency checking
operates **per-part** — it flags individual bad measurements in real time.

In [ ]:
import sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))
import config as C
from data.loaders  import load_parquet, build_cv_index, join_cv_to_parquet, load_tabular_vector
from data.dataset  import MultiModalDataset, collate_fn, get_tabular_cols, get_dxp_cols
from models.fusion import InjectionMouldingFusionModel
from training.splits import build_experiment_label_map, random_split
from analysis.consistency import (
    EmbeddingAnomalyDetector, consistency_report,
    batch_consistency_sweep, summarise_sweep, detect_flatline_channels,
)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 120
print("Device:", DEVICE)

In [ ]:
df     = load_parquet(C.PARQUET_PATH)
cv_idx = build_cv_index()
df     = join_cv_to_parquet(df, cv_idx)
tabular_cols  = get_tabular_cols(df)
dxp_cols      = get_dxp_cols(df)
exp_label_map = build_experiment_label_map(df)
train_df, _, test_df = random_split(df)

scaler   = StandardScaler()
scaler.fit(np.array([load_tabular_vector(r, tabular_cols) for _, r in train_df.iterrows()]))
tab_med  = {c: float(train_df[c].median()) for c in tabular_cols if c in train_df.columns}
roi_med  = {c: float(train_df[c].median()) for c in C.THERMAL_ROI_COLS if c in train_df.columns}

def _loader(df_sub, shuffle=False, bs=C.BATCH_SIZE):
    ds = MultiModalDataset(df_sub, tabular_cols, dxp_cols, train=False,
                           tabular_medians=tab_med, roi_medians=roi_med, tabular_scaler=scaler)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, collate_fn=collate_fn, num_workers=0)

model = InjectionMouldingFusionModel(
    tabular_in_dim=len(tabular_cols), n_dxp_channels=len(dxp_cols),
    n_experiments=len(exp_label_map),
).to(DEVICE)
ckpt = C.MODELS_DIR / "fusion_final.pt"
if ckpt.exists():
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
model.eval()
print("Model loaded. Building reference detectors...")

## 1 · Build Reference Anomaly Detectors

In [ ]:
ref_loader = _loader(train_df, bs=32)

thermal_det = EmbeddingAnomalyDetector.from_model_outputs(
    model, ref_loader, DEVICE, modality="thermal"
)
fused_det = EmbeddingAnomalyDetector.from_model_outputs(
    model, ref_loader, DEVICE, modality="fused"
)
tcn_det = EmbeddingAnomalyDetector.from_model_outputs(
    model, ref_loader, DEVICE, modality="tcn"
)
print(f"Thermal anomaly threshold: {thermal_det.threshold:.4f}  (99th pct of ref Z-score)")
print(f"Fused   anomaly threshold: {fused_det.threshold:.4f}")
print(f"TCN     anomaly threshold: {tcn_det.threshold:.4f}")

## 2 · Cross-Modal Cosine Distribution

In [ ]:
from analysis.consistency import cross_modal_cosine
import torch

cos_sims, labels_list = [], []
label_idx = C.LABEL_COLS.index(C.PRIMARY_TARGET)
test_loader = _loader(test_df, bs=16)

model.eval()
with torch.no_grad():
    for batch in test_loader:
        b = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        out = model(b)

        # Only compute for parts where both modalities are valid
        t_valid = b["thermal_valid"]
        s_valid = b["sequence_valid"]
        both    = t_valid & s_valid

        if both.any():
            cos = cross_modal_cosine(out["thermal_emb"][both], out["tcn_emb"][both])
            cos_sims.extend(cos.cpu().numpy().tolist())
            labels_list.extend(b["labels"][both, label_idx].cpu().numpy().tolist())

cos_sims = np.array(cos_sims)
labels_a = np.array(labels_list)

if len(cos_sims) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    bins = np.linspace(-1, 1, 40)

    for lbl, colour, name in [(0,"#42A5F5","OK"), (1,"#EF5350","NOK")]:
        mask = labels_a == lbl
        if mask.sum() > 0:
            ax1.hist(cos_sims[mask], bins=bins, alpha=0.65, color=colour, label=name, density=True)

    ax1.axvline(0.3,  ls="--", color="#FF9800", label="YELLOW (0.30)")
    ax1.axvline(0.1,  ls="--", color="#EF5350", label="RED (0.10)")
    ax1.set_xlabel("Cosine similarity (thermal_emb · tcn_emb)")
    ax1.set_ylabel("Density")
    ax1.set_title("Cross-Modal Cosine: OK vs NOK Parts")
    ax1.legend()

    ax2.scatter(cos_sims, labels_a + np.random.normal(0, 0.04, len(labels_a)),
                c=["#EF5350" if l else "#42A5F5" for l in labels_a],
                alpha=0.5, s=20, edgecolors="white", linewidths=0.2)
    ax2.axvline(0.3, ls="--", color="#FF9800")
    ax2.set_xlabel("Cosine similarity"); ax2.set_yticks([0, 1]); ax2.set_yticklabels(["OK","NOK"])
    ax2.set_title("Cosine vs Label (NOK parts tend to show lower cross-modal agreement)")

    plt.suptitle("Cross-Modal Consistency: Thermal ↔ Sequence Encoder Agreement", fontsize=12)
    plt.tight_layout()
    plt.savefig(C.PLOTS_DIR/"consistency_cosine.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Mean cosine (OK) : {cos_sims[labels_a==0].mean():.4f}")
    print(f"Mean cosine (NOK): {cos_sims[labels_a==1].mean():.4f}")
else:
    print("No parts with both thermal and sequence valid in test set.")

## 3 · DXP Flat-Line Simulation

In [ ]:
# Simulate a disconnected probe (zero-out one DXP channel)
test_loader_1 = _loader(test_df.head(10), bs=1)
batch_demo = next(iter(test_loader_1))
seq_normal = batch_demo["sequence"][0].cpu().numpy()   # (n_ch, T)

# Simulate probe fault on channel 0 (injection pressure)
seq_faulty = seq_normal.copy()
seq_faulty[0, :] = 0.0

ch_names = [c.replace("DXP_","") for c in dxp_cols]

fig, axes = plt.subplots(2, 1, figsize=(13, 6))
for i, (seq, title) in enumerate([(seq_normal, "Normal DXP signal"),
                                   (seq_faulty, "Faulty DXP — channel 0 flat-lined")]):
    for ci in range(min(len(dxp_cols), seq.shape[0])):
        axes[i].plot(seq[ci], alpha=0.7, linewidth=1,
                     label=ch_names[ci] if ci < 8 else None)
    axes[i].set_title(title); axes[i].legend(fontsize=7, loc="upper right", ncol=4)
plt.tight_layout(); plt.show()

# Detect flat-line
from analysis.consistency import detect_flatline_channels
normal_fl = detect_flatline_channels(seq_normal)
faulty_fl = detect_flatline_channels(seq_faulty)
n_flatline_normal = sum(normal_fl.values())
n_flatline_faulty = sum(faulty_fl.values())
print(f"Normal signal — flat-line channels: {n_flatline_normal}")
print(f"Faulty signal — flat-line channels: {n_flatline_faulty}")
faulty_chs = {ch_names[i] if i < len(ch_names) else i for i, v in faulty_fl.items() if v}
print(f"Flagged channels: {faulty_chs}")

## 4 · Batch Consistency Sweep

In [ ]:
reports = batch_consistency_sweep(
    model, test_loader, DEVICE,
    thermal_detector=thermal_det,
    tcn_detector=tcn_det,
    fused_detector=fused_det,
    dxp_cols=dxp_cols,
)
summary = summarise_sweep(reports)

print("=" * 55)
print("CONSISTENCY SWEEP SUMMARY")
print("=" * 55)
for k, v in summary.items():
    print(f"  {k:30s}: {v}")
print("=" * 55)

In [ ]:
# Fault score distribution
fault_scores = [r["fault_score"] for r in reports]
alerts       = [r["global_alert"] for r in reports]
label_flags  = []

test_loader_labels = _loader(test_df, bs=1)
for batch in test_loader_labels:
    label_flags.append(int(batch["labels"][0, C.LABEL_COLS.index(C.PRIMARY_TARGET)].item()))
label_flags = label_flags[:len(fault_scores)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

alert_colours = {"GREEN": "#66BB6A", "YELLOW": "#FF9800", "RED": "#EF5350"}
ax1.bar(range(len(fault_scores)), fault_scores,
        color=[alert_colours.get(a,"gray") for a in alerts], edgecolor="white", linewidth=0.3)
ax1.axhline(0.3, ls="--", color="#FF9800", label="YELLOW (0.30)")
ax1.axhline(0.6, ls="--", color="#EF5350", label="RED (0.60)")
ax1.set_xlabel("Part index"); ax1.set_ylabel("Fault score")
ax1.set_title("Per-Part Sensor Consistency Fault Score"); ax1.legend()

# Correlation between fault score and actual NOK label
if len(label_flags) == len(fault_scores):
    from sklearn.metrics import roc_auc_score
    try:
        auc = roc_auc_score(label_flags, fault_scores)
    except Exception:
        auc = float("nan")
    ok_scores  = [fs for fs, l in zip(fault_scores, label_flags) if l == 0]
    nok_scores = [fs for fs, l in zip(fault_scores, label_flags) if l == 1]
    ax2.boxplot([ok_scores, nok_scores], labels=["OK", "NOK"], patch_artist=True,
                boxprops=dict(facecolor="#90CAF9"),
                medianprops=dict(color="#B71C1C", linewidth=2))
    ax2.set_ylabel("Fault score")
    ax2.set_title(f"Fault Score vs NOK Label  (AUC={auc:.3f})")

plt.suptitle("Sensor Consistency: Fault Scores Across Test Set", fontsize=12)
plt.tight_layout()
plt.savefig(C.PLOTS_DIR/"consistency_fault_scores.png", dpi=150, bbox_inches="tight")
plt.show()